# Phase 5: Trend Analysis and Correlation Analysis

## Purpose

This notebook analyzes sales trends over time and explores relationships between numerical variables in the retail orders dataset.

Project 1 focused on cleaning and validating the dataset.  
Project 2 focuses on Exploratory Data Analysis.

In this phase, the goal is to understand how revenue, order volume, and average order value change over time, and whether numerical columns such as Quantity, UnitPrice, ItemsInCart, and TotalPrice are related.

Main questions:

- How does revenue change over time?
- Which months have the highest and lowest sales?
- How does order volume change over time?
- Which numerical variables are strongly related?
- What business meaning can we get from these patterns?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
df = pd.read_csv("../data/processed/ecommerce_orders_project2_eda_ready.csv")

df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,HasCoupon,TotalPrice_Outlier_IQR
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,Coupon Used,Normal
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,Coupon Used,Normal
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,Coupon Used,Normal
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,Coupon Used,Normal
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,Coupon Used,Normal


In [3]:
df["Date"] = pd.to_datetime(df["Date"])

df["Date"].head()

0   2023-01-04
1   2024-08-23
2   2024-02-27
3   2023-10-15
4   2025-05-08
Name: Date, dtype: datetime64[ns]

In [4]:
#create time based columns
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["MonthName"] = df["Date"].dt.month_name()
df["YearMonth"] = df["Date"].dt.to_period("M").astype(str)

df[["Date", "Year", "Month", "MonthName", "YearMonth"]].head()

,Date,Year,Month,MonthName,YearMonth
0,2023-01-04,2023,1,January,2023-01
1,2024-08-23,2024,8,August,2024-08
2,2024-02-27,2024,2,February,2024-02
3,2023-10-15,2023,10,October,2023-10
4,2025-05-08,2025,5,May,2025-05


In [5]:
#create figures folder
os.makedirs("../reports/figures", exist_ok=True)

print("Figures folder is ready.")

Figures folder is ready.


In [6]:
#monthly revenue trend
monthly_revenue = (
    df.groupby("YearMonth")["TotalPrice"]
    .sum()
    .reset_index()
    .rename(columns={"TotalPrice": "MonthlyRevenue"})
)

monthly_revenue.head()

,YearMonth,MonthlyRevenue
0,2023-01,56685.75
1,2023-02,40117.66
2,2023-03,48609.37
3,2023-04,27751.71
4,2023-05,63836.84


In [ ]:
#plot monthly revenue trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_revenue["YearMonth"], monthly_revenue["MonthlyRevenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/monthly_revenue_trend.png")
plt.show()

In [ ]:
#monthly order volume trend
monthly_orders = (
    df.groupby("YearMonth")["OrderID"]
    .nunique()
    .reset_index()
    .rename(columns={"OrderID": "MonthlyOrders"})
)

monthly_orders.head()

In [ ]:
#plot monthly order volume trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_orders["YearMonth"], monthly_orders["MonthlyOrders"], marker="o")
plt.title("Monthly Order Volume Trend")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/monthly_order_volume_trend.png")
plt.show()

In [ ]:
#monthly average order value trend
monthly_aov = (
    df.groupby("YearMonth")["TotalPrice"]
    .mean()
    .reset_index()
    .rename(columns={"TotalPrice": "AverageOrderValue"})
)

monthly_aov.head()

In [ ]:
#plot monthly average order value trend
plt.figure(figsize=(12, 6))
plt.plot(monthly_aov["YearMonth"], monthly_aov["AverageOrderValue"], marker="o")
plt.title("Monthly Average Order Value Trend")
plt.xlabel("Month")
plt.ylabel("Average Order Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/monthly_aov_trend.png")
plt.show()

In [ ]:
#monthly trend summary
monthly_trend_summary = monthly_revenue.merge(monthly_orders, on="YearMonth")
monthly_trend_summary = monthly_trend_summary.merge(monthly_aov, on="YearMonth")


In [ ]:
monthly_trend_summary["MonthlyRevenue"] = monthly_trend_summary["MonthlyRevenue"].round(2)
monthly_trend_summary["AverageOrderValue"] = monthly_trend_summary["AverageOrderValue"].round(2)

monthly_trend_summary.head()

In [ ]:
#highest and lowest revenue months
highest_revenue_month = monthly_trend_summary.loc[
    monthly_trend_summary["MonthlyRevenue"].idxmax()
]

lowest_revenue_month = monthly_trend_summary.loc[
    monthly_trend_summary["MonthlyRevenue"].idxmin()
]

print("Highest Revenue Month:")
print(highest_revenue_month)

print("\nLowest Revenue Month:")
print(lowest_revenue_month)

In [ ]:
monthly_trend_summary.to_csv("../reports/monthly_trend_summary.csv", index=False)

print("Monthly trend summary saved successfully.")

In [ ]:
#correlation analysis
#numerical columns
numeric_columns = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

df[numeric_columns].head()

In [ ]:
#correlation matrix
correlation_matrix = df[numeric_columns].corr().round(2)

correlation_matrix

In [ ]:
correlation_matrix.to_csv("../reports/correlation_matrix.csv")

print("Correlation matrix saved successfully.")

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45)
plt.yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
plt.title("Correlation Matrix Heatmap")

for i in range(len(correlation_matrix.index)):
    for j in range(len(correlation_matrix.columns)):
        plt.text(j, i, correlation_matrix.iloc[i, j], ha="center", va="center")

plt.tight_layout()
plt.savefig("../reports/figures/correlation_matrix_heatmap.png")
plt.show()

In [ ]:
#Scatter plot: UnitPrice vs TotalPrice
plt.figure(figsize=(8, 6))
plt.scatter(df["UnitPrice"], df["TotalPrice"], alpha=0.6)
plt.title("UnitPrice vs TotalPrice")
plt.xlabel("UnitPrice")
plt.ylabel("TotalPrice")
plt.tight_layout()
plt.savefig("../reports/figures/unitprice_vs_totalprice.png")
plt.show()

In [ ]:
#Scatter plot: Quantity vs TotalPrice
plt.figure(figsize=(8, 6))
plt.scatter(df["Quantity"], df["TotalPrice"], alpha=0.6)
plt.title("Quantity vs TotalPrice")
plt.xlabel("Quantity")
plt.ylabel("TotalPrice")
plt.tight_layout()
plt.savefig("../reports/figures/quantity_vs_totalprice.png")
plt.show()

In [ ]:
#Scatter plot: ItemsInCart vs TotalPrice
plt.figure(figsize=(8, 6))
plt.scatter(df["ItemsInCart"], df["TotalPrice"], alpha=0.6)
plt.title("ItemsInCart vs TotalPrice")
plt.xlabel("ItemsInCart")
plt.ylabel("TotalPrice")
plt.tight_layout()
plt.savefig("../reports/figures/itemsincart_vs_totalprice.png")
plt.show()

In [ ]:
correlation_pairs = []

for i in range(len(numeric_columns)):
    for j in range(i + 1, len(numeric_columns)):
        col1 = numeric_columns[i]
        col2 = numeric_columns[j]
        corr_value = correlation_matrix.loc[col1, col2]
        
        correlation_pairs.append({
            "Variable 1": col1,
            "Variable 2": col2,
            "Correlation": corr_value
        })

correlation_pairs_df = pd.DataFrame(correlation_pairs)
correlation_pairs_df = correlation_pairs_df.sort_values(by="Correlation", ascending=False)

correlation_pairs_df

In [ ]:
correlation_pairs_df.to_csv("../reports/correlation_pairs_summary.csv", index=False)

print("Correlation pairs summary saved successfully.")

In [ ]:
df.to_csv("../data/processed/ecommerce_orders_project2_trend_correlation_ready.csv", index=False)

print("Trend and correlation ready dataset saved successfully.")

## Trend and Correlation Analysis Summary

This phase analyzed time-based trends and relationships between numerical variables in the retail orders dataset.

Project 1 focused on data cleaning and validation.  
Project 2 focuses on Exploratory Data Analysis, so this phase was used to discover sales trends and numerical relationships.

### Trend Analysis

Monthly revenue, monthly order volume, and monthly average order value were analyzed.

The highest revenue month was **2024-06**, with:

- Monthly Revenue: 68,068.54
- Monthly Orders: 53
- Average Order Value: 1,284.31

The lowest revenue month was **2023-04**, with:

- Monthly Revenue: 27,751.71
- Monthly Orders: 31
- Average Order Value: 895.22

### Business Meaning of Trend Analysis

The highest revenue month had both strong order volume and a high average order value.

The lowest revenue month had fewer orders and a lower average order value.

This shows that monthly revenue is affected by both:

- Number of orders
- Average value of each order

So, increasing revenue is not only about getting more orders. It is also about increasing the value of each order.

### Correlation Analysis

The numerical columns analyzed were:

- Quantity
- UnitPrice
- ItemsInCart
- TotalPrice

The strongest relationships found were:

- UnitPrice and TotalPrice: 0.72
- Quantity and ItemsInCart: 0.65
- Quantity and TotalPrice: 0.62
- ItemsInCart and TotalPrice: 0.39

### Business Meaning of Correlation Analysis

UnitPrice has the strongest relationship with TotalPrice.  
This means higher-priced products are strongly connected with higher order values.

Quantity also has a strong relationship with TotalPrice.  
This means customers who buy more units usually create higher-value orders.

ItemsInCart has a moderate relationship with TotalPrice.  
This means having more items in the cart may increase order value, but it is not as strong as UnitPrice or Quantity.

### Important Note About Correlation

Correlation does not prove cause and effect.

For example, UnitPrice and TotalPrice have a strong positive correlation, but this does not mean UnitPrice is the only reason TotalPrice increases.

Correlation should be treated as a clue for deeper business analysis.

### Conclusion

This phase helped identify important sales trends and numerical relationships.

The dataset is now ready for the next EDA phase:

Product, payment method, coupon, referral source, and order status analysis.